# 식물과 인간: 붓꽃 종 분류 실습

코드를 위에서 아래로 **한 칸씩 실행**해 보세요. 이 모델은 꽃 사진이 아닌 꽃잎과 꽃받침의 측정값을 사용합니다.

## 시작 전 준비

오류가 발생하면 VS Code의 터미널에서 `python -m pip install scikit-learn matplotlib`를 실행하세요.

In [1]:
"""식물과 인간: 붓꽃 3종 분류 실습.

실행: python iris_project_practice.py
설치: python -m pip install scikit-learn matplotlib

붓꽃의 꽃받침/꽃잎 길이·너비(cm)만 사용하며 실제 사진 인식은 하지 않습니다.
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


# 한글 그래프가 깨지는 것을 방지합니다(설치된 글꼴 중 하나를 선택).
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in installed_fonts:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

# 보고서에 사용하는 한글 이름과 일치시킵니다.
KOREAN_NAMES = ["부채붓꽃", "북방푸른꽃창포", "버지니아꽃창포"]
FEATURE_NAMES = ["꽃받침 길이", "꽃받침 너비", "꽃잎 길이", "꽃잎 너비"]
RESULT_DIR = Path.cwd() / "iris_results"
RESULT_DIR.mkdir(exist_ok=True)




## 1단계: 데이터 확인

붓꽃 150개의 꽃받침과 꽃잎 측정값을 불러오고 세 종의 평균 크기를 확인합니다.

In [2]:
# 1. 150개의 붓꽃 측정값 불러오기
iris = load_iris()
X = iris.data      # 150행 x 4열, 측정값(단위: cm)
y = iris.target    # 각 행의 정답: 0, 1, 2
print("=== 1. 데이터 확인 ===")
print("총 데이터:", len(X), "개")
print("측정 항목:", ", ".join(FEATURE_NAMES))
for species in range(3):
    flower_data = X[y == species]
    print(f"{KOREAN_NAMES[species]}: {len(flower_data)}개, "
          f"꽃잎 평균 길이 {flower_data[:, 2].mean():.2f}cm, "
          f"꽃잎 평균 너비 {flower_data[:, 3].mean():.2f}cm")




=== 1. 데이터 확인 ===
총 데이터: 150 개
측정 항목: 꽃받침 길이, 꽃받침 너비, 꽃잎 길이, 꽃잎 너비
부채붓꽃: 50개, 꽃잎 평균 길이 1.46cm, 꽃잎 평균 너비 0.25cm
북방푸른꽃창포: 50개, 꽃잎 평균 길이 4.26cm, 꽃잎 평균 너비 1.33cm
버지니아꽃창포: 50개, 꽃잎 평균 길이 5.55cm, 꽃잎 평균 너비 2.03cm


## 2단계: 데이터 나누기

120개는 학습용으로, 30개는 최종 시험용으로 남겨 둡니다.

In [3]:
# 2. 학습용 120개 / 시험용 30개 분리 (종마다 비율 동일)
# 시험용 데이터는 학습과 교차검증에 사용하지 않습니다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("\n=== 2. 데이터 나누기 ===")
print(f"학습용: {len(X_train)}개 / 시험용: {len(X_test)}개")





=== 2. 데이터 나누기 ===
학습용: 120개 / 시험용: 30개


## 3단계: AI 모델 두 개 학습

의사결정나무와 KNN을 같은 조건에서 비교합니다. 결과가 보고서와 일치하는지 확인합니다.

In [4]:
# 3. 이해하기 쉬운 두 모델 준비
# 의사결정나무: 길이/너비를 기준으로 질문하며 분류
# KNN: 네 측정값이 비슷한 꽃 5개를 참고해 분류
models = {
    "의사결정나무": DecisionTreeClassifier(max_depth=3, random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\n=== 3. 모델별 정확도 ===")

results = {}
for name, model in models.items():
    # 학습용 데이터를 5묶음으로 나눠 번갈아 검증
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    # 전체 학습용 120개를 학습한 다음 시험용 30개를 예측
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    correct = int(np.sum(predictions == y_test))
    results[name] = {"cv": float(cv_scores.mean()), "test": float(accuracy)}
    print(f"{name}: 교차검증 {cv_scores.mean():.2%}, "
          f"시험 {correct}/30개 정답 ({accuracy:.2%})")





=== 3. 모델별 정확도 ===
의사결정나무: 교차검증 95.83%, 시험 29/30개 정답 (96.67%)
KNN: 교차검증 95.83%, 시험 28/30개 정답 (93.33%)


## 4단계: 틀린 예측 찾기

예측을 틀린 꽃 한 개를 찾아 네 가지 측정값을 확인합니다.

In [5]:
# 4. 의사결정나무가 틀린 사례 조사
# 이는 'AI가 어떤 붓꽃을 어려워했는가?'를 보고서에 기록하기 위한 과정입니다.
print("\n=== 4. 틀린 예측 조사(의사결정나무) ===")
tree = models["의사결정나무"]
tree_predictions = tree.predict(X_test)
errors = np.flatnonzero(tree_predictions != y_test)
if len(errors) == 0:
    print("이번 시험 데이터에서는 틀린 예측이 없습니다.")
for i in errors:
    measurements = ", ".join(f"{v:.1f}cm" for v in X_test[i])
    print(f"실제: {KOREAN_NAMES[y_test[i]]} / "
          f"예측: {KOREAN_NAMES[tree_predictions[i]]}")
    print(f"측정값(꽃받침 길이/너비, 꽃잎 길이/너비): {measurements}")

print("혼동행렬(행=실제 종, 열=예측 종):")
print(confusion_matrix(y_test, tree_predictions))





=== 4. 틀린 예측 조사(의사결정나무) ===
실제: 북방푸른꽃창포 / 예측: 버지니아꽃창포
측정값(꽃받침 길이/너비, 꽃잎 길이/너비): 6.7cm, 3.0cm, 5.0cm, 1.7cm
혼동행렬(행=실제 종, 열=예측 종):
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]


## 5단계: 어느 부분이 더 중요할까?

꽃받침만 사용할 때, 꽃잎만 사용할 때, 모두 사용할 때의 결과를 비교합니다.

In [6]:
# 5. 꽃잎과 꽃받침 중 어느 쪽이 분류에 더 도움이 되었는가?
print("\n=== 5. 측정 부위별 교차검증 비교 ===")
feature_sets = {
    "꽃받침만": [0, 1],
    "꽃잎만": [2, 3],
    "네 특징 모두": [0, 1, 2, 3],
}
feature_results = {}
for label, columns in feature_sets.items():
    temp_model = DecisionTreeClassifier(max_depth=3, random_state=42)
    scores = cross_val_score(temp_model, X_train[:, columns], y_train,
                             cv=cv, scoring="accuracy")
    feature_results[label] = float(scores.mean())
    print(f"{label}: {scores.mean():.2%}")





=== 5. 측정 부위별 교차검증 비교 ===
꽃받침만: 80.83%
꽃잎만: 95.83%
네 특징 모두: 95.83%


## 6단계: 그래프 만들기

붓꽃 종별 꽃잎 크기 분포와 부위별 분류 성능 그래프를 저장합니다.

In [7]:
# 6. 보고서에 사용할 그래프 자동 저장
fig, ax = plt.subplots(figsize=(8, 5))
for i, species_name in enumerate(KOREAN_NAMES):
    points = X[y == i]
    ax.scatter(points[:, 2], points[:, 3], label=species_name, alpha=0.75, s=30)
ax.set(xlabel="꽃잎 길이(cm)", ylabel="꽃잎 너비(cm)",
       title="붓꽃 3종의 꽃잎 크기 분포")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(RESULT_DIR / "01_꽃잎_분포.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
labels = list(feature_results.keys())
values = [feature_results[label] * 100 for label in labels]
bars = ax.bar(labels, values)
ax.set(ylabel="교차검증 평균 정확도(%)", ylim=(0, 110),
       title="꽃받침과 꽃잎의 분류 성능 비교")
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 1,
            f"{value:.2f}%", ha="center")
fig.tight_layout()
fig.savefig(RESULT_DIR / "02_측정_부위_비교.png", dpi=180)
plt.close(fig)
print(f"\n그래프 저장 위치: {RESULT_DIR}")





그래프 저장 위치: c:\Users\pc\Documents\iris-species-classification\iris_results


## 7단계: 새 꽃 예측

꽃받침과 꽃잎의 길이 및 너비 네 숫자를 직접 바꿔 예측 결과를 확인해 볼 수 있습니다.

In [8]:
# 7. 새 측정값 한 개로 예측해 보기
# 값 입력 순서: 꽃받침 길이, 꽃받침 너비, 꽃잎 길이, 꽃잎 너비(cm)
new_flower = [[5.1, 3.5, 1.4, 0.2]]
answer = int(tree.predict(new_flower)[0])
print("\n=== 6. 측정값으로 종 예측하기 ===")
print("입력:", new_flower[0])
print("예측:", KOREAN_NAMES[answer], f"({iris.target_names[answer]})")
print("※ 사진을 직접 분류하는 모델이 아니라 네 가지 측정값을 분류하는 모델입니다.")



=== 6. 측정값으로 종 예측하기 ===
입력: [5.1, 3.5, 1.4, 0.2]
예측: 부채붓꽃 (setosa)
※ 사진을 직접 분류하는 모델이 아니라 네 가지 측정값을 분류하는 모델입니다.
